# Lesson 01: BM25 Keyword Search
> **Goal:** Learn how BM25 scores each document for a search query and returns the best results.

---

### The BM25 Formula
BM25 gives every document $D$ a numerical score for a query $Q$:

$$\text{Score}(D, Q) = \sum_{\text{word} \in Q} \text{IDF}(\text{word}) \cdot \frac{\text{TF}(\text{word}, D) \cdot (k_1 + 1)}{\text{TF}(\text{word}, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

### In Plain English (The 3 Ingredients):
1. **TF (Term Frequency):** How many times query words appear in the document.
2. **IDF (Inverse Document Frequency):** Rare words (like `SKU-9921`) give more points; common words (like `the`) give fewer points.
3. **Length Normalization ($b \approx 0.75$):** Stops long documents from winning just because they have more words.


In [1]:
# 1. Load documents from data.txt (robust path lookup)
from pathlib import Path

for path in [Path("data.txt"), Path("src/notebooks/data.txt"), Path("../data.txt")]:
    if path.exists():
        data_file = path
        break
else:
    raise FileNotFoundError("Could not find data.txt")

with open(data_file, "r", encoding="utf-8") as f:
    documents = [p.strip() for p in f.read().split("\n\n") if p.strip()]

print(f"Loaded {len(documents)} document paragraphs.")
print(f"Sample Doc #1: \"{documents[0][:90]}...\"")


Loaded 30 document paragraphs.
Sample Doc #1: "Aura Technologies (NASDAQ: ATRX) was founded in 2018 by Chief Executive Officer Elena Vanc..."


In [2]:
# 2. The BM25 Scoring Function (~15 lines)
import math, re
from collections import Counter

# Simple tokenization (keeps letters, numbers, and hyphens for codes like SKU-9921)
def tokenize(text: str) -> list[str]:
    return re.findall(r"[\w-]+", text.lower())

tokenized_docs = [tokenize(doc) for doc in documents]
N = len(documents)
avg_len = sum(len(d) for d in tokenized_docs) / N

# Precompute word rarity (IDF)
doc_counts = Counter(word for doc in tokenized_docs for word in set(doc))
idf = {word: math.log((N - count + 0.5) / (count + 0.5) + 1.0) for word, count in doc_counts.items()}

# BM25 Formula implementation
def bm25_score(query: str, doc_tokens: list[str], k1: float = 1.5, b: float = 0.75) -> float:
    doc_len = len(doc_tokens)
    tf = Counter(doc_tokens)
    length_penalty = (1 - b) + b * (doc_len / avg_len)
    
    score = 0.0
    for word in tokenize(query):
        if word in idf:
            # Direct BM25 formula
            term_score = idf[word] * (tf[word] * (k1 + 1)) / (tf[word] + k1 * length_penalty)
            score += term_score
    return score

print("BM25 scoring formula ready!")


BM25 scoring formula ready!


In [3]:
# 3. Score every document for a query and see the scores
query = "Titan-X9 SKU-9921"
print(f"Scoring all {len(documents)} documents for query: '{query}'\n")

# Compute BM25 score for each document
doc_scores = []
for idx, doc_tokens in enumerate(tokenized_docs):
    score = bm25_score(query, doc_tokens)
    doc_scores.append((idx + 1, score))

# Sort by highest score
doc_scores.sort(key=lambda x: x[1], reverse=True)

# Display the top 5 document scores
print(f"{'Rank':<6} | {'Document':<10} | {'BM25 Score':<12}")
print("-" * 32)
for rank, (doc_num, score) in enumerate(doc_scores[:5], 1):
    print(f"{rank:<6} | Doc #{doc_num:<5} | {score:.2f}")

print(f"\nDoc #{doc_scores[0][0]} has the highest score ({doc_scores[0][1]:.2f}) and wins!")


Scoring all 30 documents for query: 'Titan-X9 SKU-9921'

Rank   | Document   | BM25 Score  
--------------------------------
1      | Doc #23    | 4.68
2      | Doc #24    | 2.61
3      | Doc #14    | 2.03
4      | Doc #13    | 1.96
5      | Doc #7     | 1.90

Doc #23 has the highest score (4.68) and wins!


In [4]:
# 4. Search Function: Return top-k matching documents
def search_bm25(query: str, top_k: int = 2) -> list[dict]:
    scored = []
    for idx, doc_tokens in enumerate(tokenized_docs):
        score = bm25_score(query, doc_tokens)
        scored.append({
            "doc_id": idx + 1,
            "score": score,
            "text": documents[idx]
        })
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

# Test exact keyword queries
queries = [
    "What is the catalog part number for the Titan-X9 processor?",
    "What was the bug incident code logged for the overheating issue?"
]

print("=== BM25 Search Results ===\n")
for q in queries:
    top_result = search_bm25(q, top_k=1)[0]
    print(f"Query: \"{q}\"")
    print(f"-> Best Match: Doc #{top_result['doc_id']} (Score: {top_result['score']:.2f})")
    print(f"   \"{top_result['text']}\"\n")


=== BM25 Search Results ===

Query: "What is the catalog part number for the Titan-X9 processor?"
-> Best Match: Doc #23 (Score: 16.12)
   "On April 15, 2024, Aura officially commenced commercial availability of the Titan-X9 neural processor, distributed through global supply chains under catalog part number SKU-9921."

Query: "What was the bug incident code logged for the overheating issue?"
-> Best Match: Doc #25 (Score: 6.38)
   "Shortly after deployment, enterprise customers reported unexpected thermal throttling when ambient temperatures exceeded 45 degrees Celsius, cataloged in engineering logs as bug incident #BUG-404."



In [5]:
# 5. Where BM25 Fails: Vocabulary Mismatch (Synonyms)
# If query words don't match document text letter-for-letter, BM25 fails!
query_synonym = "What technology allows devices to run without power grid connectivity?"
results = search_bm25(query_synonym, top_k=3)

print(f"Query: \"{query_synonym}\"\n")
print("Top BM25 Results:")
for rank, res in enumerate(results, 1):
    print(f"  Rank {rank} (Score: {res['score']:.2f}) -> Doc #{res['doc_id']}: {res['text'][:70]}...")

# Check where the true answer (Doc #11 - GreenVolt) was ranked safely
all_docs = search_bm25(query_synonym, top_k=len(documents))
greenvolt_rank = next((r for r, item in enumerate(all_docs, 1) if item['doc_id'] == 11), "Not in results")

print(f"\nNotice: Doc #11 (GreenVolt) ranked #{greenvolt_rank} out of {len(documents)}!")
print("Reason: Doc #11 uses the words 'energy-harvesting off-grid', not 'power grid'.")
print("BM25 cannot match meaning without exact words -> That is why we need Embeddings (Lesson 02)!")


Query: "What technology allows devices to run without power grid connectivity?"

Top BM25 Results:
  Rank 1 (Score: 3.92) -> Doc #6: Dr. Aris Thorne, a distinguished computer architect from Stanford Univ...
  Rank 2 (Score: 3.33) -> Doc #11: In October 2023, Aura publicly revealed an environmental sustainabilit...
  Rank 3 (Score: 2.52) -> Doc #12: Concurrently, Aura’s legal department successfully resolved an intelle...

Notice: Doc #11 (GreenVolt) ranked #2 out of 30!
Reason: Doc #11 uses the words 'energy-harvesting off-grid', not 'power grid'.
BM25 cannot match meaning without exact words -> That is why we need Embeddings (Lesson 02)!


---
### Summary for Students
1. **The Formula:** Scores each document by matching words, weighting by rarity (IDF), and penalizing document length.
2. **Scores:** Every document gets a number. The document with the highest score is Rank 1.
3. **Exact Matches:** Unbeatable for catalog numbers (`SKU-9921`) and bug codes (`#BUG-404`).
4. **The Limit:** Blind to synonyms and meaning (`power grid` vs `energy-harvesting`).
5. **Next Step:** In **Lesson 02**, we introduce **Vector Embeddings** to search by *meaning* rather than exact words!
